# Projeto 04 — Agentes Inteligentes

**Disciplina:** GCC 128 — Inteligência Artificial  
**Instituição:** Universidade Federal de Lavras (UFLA)  
**Docentes:** Prof. Ahmed Ali Abdalla Esmin  
**Discentes:** Caio Bueno Finocchio Martins e Lana da Silva Miranda


## 1. Visão geral

O projeto cria revisões personalizadas a partir do conteúdo estudado e das preferências do usuário, incluindo prioridade (qual tópico do conteúdo é mais relevante), profundidade (o quão difícil devem ser as questões), foco (qual aspecto do conteúdo foi estudado) e nível de detalhe da revisão.
Dois agentes, Tutor e Avaliador, produzem uma revisão, questões e um gabarito para conferência. O Avaliador utiliza o texto produzido pelo Tutor, estabelecendo uma colaboração entre os agentes.

Cada pedido de revisão é independente, por isso não precisamos guardar o histórico de pedidos anteriores. O `MessagesState` reúne apenas as mensagens da execução atual; não usamos memória persistente nem `checkpointer`. Também não foi necessário usar um Supervisor porque o fluxo de tarefas é único e não há necessidade de escolher qual agente deve atuar a cada momento.


## 2. Instalação e configuração

### 2.1 Dependências

Usamos `langgraph` para o fluxo multiagente, `langchain` para os componentes de mensagens e ferramentas e `langchain-groq` para acessar o modelo, conforme usado nas aulas.
O ambiente do notebook também precisa ter `IPython` e `ipywidgets` disponíveis para apresentar os resultados.

In [ ]:
!pip install -q langgraph langchain langchain-groq

### 2.2 Chave da API Groq

Usamos `getpass` para solicitar a chave sem exibi-la durante a digitação ou deixá-la escrita no notebook.

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")

### 2.3 Configuração dos modelos

Escolhemos o `openai/gpt-oss-20b` pelo acesso gratuito oferecido pela Groq no desenvolvimento do projeto.
O Tutor usa temperatura 0.2 para uma revisão mais controlada; o Avaliador usa 0.4 para permitir alguma variedade nas questões.
Mantivemos ambas baixas para favorecer a consistência das respostas.

In [ ]:
from langchain_groq import ChatGroq

llm_tutor = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)
llm_avaliador = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

## 3. Preparação das opções de revisão

### 3.1 Focos permitidos

Mantivemos o conteúdo como texto livre para atender a diferentes disciplinas, mas limitamos os focos a seis categorias.
Assim, o modelo seleciona opções adequadas sem inventar categorias: código, por exemplo, faz sentido para programação, mas pode não servir para algum assunto de história.

In [ ]:
import json

FOCOS_POSSIVEIS = [
    "conceitual",
    "comparacao",
    "aplicacao",
    "interpretacao",
    "codigo",
    "resolucao_de_problemas"
]

### 3.2 Análise do conteúdo com `gerar_opcoes`

`gerar_opcoes` identifica tópicos e seleciona focos relacionados ao conteúdo, evitando oferecer sugestões genéricas.
Essa análise reutiliza o modelo do Tutor como preparação da entrada; não é um terceiro agente.

In [ ]:
def gerar_opcoes(conteudo: str):

    prompt = f"""
    Analise o conteúdo estudado abaixo.

    CONTEÚDO:
    {conteudo}

    Identifique:
    1. Os principais tópicos estudados que podem ser escolhidos como prioridade.
    2. Quais focos de revisão são adequados para esse conteúdo.

    Os focos só podem ser escolhidos desta lista:
    {FOCOS_POSSIVEIS}

    Por exemplo, não selecione "codigo" se o conteúdo não envolver programação.

    Retorne SOMENTE um JSON neste formato:

    {{
        "topicos": ["topico 1", "topico 2"],
        "focos": ["foco 1", "foco 2"]
    }}
    """

    resposta = llm_tutor.invoke(prompt)

    return json.loads(resposta.content)

### 3.3 Significado das preferências

Separamos as preferências para que o usuário possa ajustar aspectos diferentes da revisão:

- **Prioridade:** qual tópico deve receber mais atenção, como `dynamic e guided` em um conteúdo sobre OpenMP.
- **Foco:** como trabalhar o conhecimento, por exemplo, comparando conceitos ou aplicando-os a uma situação.
- **Profundidade:** nível de exigência das questões - básica, intermediária ou avançada.
- **Tamanho:** extensão da revisão - curta, média ou detalhada.

Profundidade e tamanho são independentes: é possível pedir uma revisão curta e questões avançadas.

### 3.4 Teste isolado do pré-processamento

Usamos um conteúdo sobre OpenMP para conferir os tópicos e focos sugeridos antes de executar o sistema completo.

In [ ]:
opcoes = gerar_opcoes(
    "OpenMP: parallel for, schedule static, dynamic e guided"
)

print(opcoes)

## 4. Ferramenta de estratégia de avaliação

### 4.1 Definição de `configurar_estrategia_avaliacao`

Criamos uma ferramenta interna para fixar as regras da avaliação, em vez de deixar o modelo decidir sozinho a quantidade e a dificuldade das questões.

A profundidade determina a dificuldade e a distribuição entre questões abertas e de múltipla escolha; o foco orienta a abordagem.
A prioridade fica com o Avaliador, pois muda os tópicos cobrados, não a estrutura da avaliação.

In [ ]:
from langchain_core.tools import tool

@tool
def configurar_estrategia_avaliacao(
        profundidade: str,
        foco: str,
    ) -> dict:
    """
    Define a estratégia de avaliação com base na profundidade e no foco escolhidos pelo usuário.

    Use esta ferramenta antes de gerar as questões da revisão. Ela retorna uma estrutura
    com orientações como quantidade e tipo de questões, nível de dificuldade e abordagem
    que deve ser priorizada.
    A profundidade define a dificuldade das questões e quantidade de abertas/fechadas. O foco
    determina a abordagem, ou seja, qual aspecto do conteúdo deve ser priorizado.

    """
    profundidade = profundidade.lower()
    foco = foco.lower()

    # Profundidade
    if profundidade == "basica":
        estrategia = {
            "multipla_escolha": 4,
            "abertas": 1,
            "dificuldade": "baixa"
        }

    elif profundidade == "intermediaria":
        estrategia = {
            "multipla_escolha": 3,
            "abertas": 2,
            "dificuldade": "media"
        }

    elif profundidade == "avancada":
        estrategia = {
            "multipla_escolha": 2,
            "abertas": 3,
            "dificuldade": "alta"
        }

    else:
        return {"erro": "Profundidade inválida"}

    # Foco
    if foco == "conceitual":
        estrategia["abordagem"] = (
            "Priorizar definições, características, relações entre conceitos "
            "e compreensão teórica."
        )

    elif foco == "comparacao":
        estrategia["abordagem"] = (
            "Priorizar diferenças, semelhanças, vantagens, desvantagens "
            "e comparação entre conceitos."
        )

    elif foco == "aplicacao":
        estrategia["abordagem"] = (
            "Priorizar situações práticas em que o estudante precise aplicar "
            "o conteúdo estudado."
        )

    elif foco == "interpretacao":
        estrategia["abordagem"] = (
            "Priorizar análise e interpretação de exemplos, situações, "
            "resultados ou informações apresentadas."
        )

    elif foco == "codigo":
        estrategia["abordagem"] = (
            "Priorizar leitura, interpretação, correção e produção de "
            "pequenos trechos de código."
        )

    elif foco == "resolucao_de_problemas":
        estrategia["abordagem"] = (
            "Priorizar problemas que exijam raciocínio e combinação dos "
            "conceitos estudados para chegar a uma solução."
        )


    return estrategia

### 4.2 Teste isolado da ferramenta

Este teste confere a estratégia para profundidade intermediária e foco em aplicação, sem executar os agentes.

In [ ]:
print(configurar_estrategia_avaliacao.invoke({
    "profundidade": "intermediaria",
    "foco": "aplicacao"
}))

## 5. Configuração dos agentes

### 5.1 Ferramenta disponível ao Avaliador

Apenas o Avaliador recebe a ferramenta, pois ela trata das questões. O Tutor não precisa dela para produzir a revisão.

In [ ]:
tools = [configurar_estrategia_avaliacao]

avaliador_com_ferramentas = llm_avaliador.bind_tools(tools)

### 5.2 Agente Tutor

As instruções do Tutor limitam a extensão do texto e destacam a prioridade escolhida.
A intenção é retomar algo já estudado, sem transformar a revisão em uma aula completa ou incluir questões nessa etapa.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

def tutor(state: MessagesState):

    instrucao = SystemMessage(content="""
    Você é um tutor responsável por produzir uma revisão do conteúdo estudado. Independente
    do tamanho solicitado lembre-se que você deve produzir uma revisão e não uma aula para
    aprender o conteúdo do zero.

    Respeite o tamanho de revisão solicitado:
    - curta: no máximo 120 palavras, destacando apenas os conceitos essenciais.
        Use no máximo 3 tópicos. Não inclua tabelas nem exemplos extensos.
    - media: entre 200 e 300 palavras, podendo incluir um exemplo curto.
    - detalhada: até 500 palavras, com explicações e exemplos quando úteis.

    Dê maior atenção ao tópico indicado como prioridade.
    Não crie questões; sua função é apenas realizar a revisão.
    """)

    resposta = llm_tutor.invoke(
        [instrucao] + state["messages"]
    )

    return {
        "messages": [resposta]
    }



### 5.3 Agente Avaliador

As instruções exigem o uso da ferramenta antes de elaborar as questões.
Também pedimos os blocos `<QUESTOES>` e `<GABARITO>` separados para que as respostas possam ser apresentadas em outro momento.

In [ ]:
def avaliador(state: MessagesState):

    instrucao = SystemMessage(content="""
    Você é um avaliador responsável por criar questões de revisão
    com base na revisão produzida pelo Tutor e nas preferências do usuário.

    Antes de gerar as questões, use a ferramenta disponível
    para definir a estratégia de avaliação.

    Siga a estratégia retornada pela ferramenta quanto a:
    - quantidade de questões;
    - nível de dificuldade;
    - tipo de abordagem;
    - foco da avaliação.

    Dê maior atenção ao tópico indicado como prioridade.

    Gere questões de múltipla escolha e abertas conforme a estratégia.
    Não refaça a revisão teórica.

    Gere mas não revele as respostas junto das questões.
    Na resposta final, use obrigatoriamente este formato:

    <QUESTOES>
    Questões para o estudante, sem respostas ou gabarito.
    </QUESTOES>

    <GABARITO>
    Respostas corretas das questões de múltipla escolha e
    respostas esperadas das questões abertas.
    </GABARITO>
    """)

    resposta = avaliador_com_ferramentas.invoke(
        [instrucao] + state["messages"]
    )

    return {
        "messages": [resposta]
    }



### 5.4 Execução da ferramenta com `ToolNode`

Usamos o `ToolNode` para executar a ferramenta dentro do fluxo do LangGraph, sem criar um mecanismo próprio para isso.

In [ ]:
tool_node = ToolNode(tools)

## 6. Construção do grafo

### 6.1 Nós e transições

O fluxo segue a ordem necessária para produzir a revisão e depois as questões:

**Início → Tutor → Avaliador → ferramenta → Avaliador → fim.**

O retorno ao Avaliador permite que ele use a estratégia definida pela ferramenta antes de concluir a avaliação.

In [ ]:
grafo = StateGraph(MessagesState)

grafo.add_node("tutor", tutor)
grafo.add_node("avaliador", avaliador)
grafo.add_node("tools", tool_node)

grafo.add_edge(START, "tutor")
grafo.add_edge("tutor", "avaliador")

grafo.add_conditional_edges(
    "avaliador",
    tools_condition
)

grafo.add_edge("tools", "avaliador")

app = grafo.compile()
print("✅ Grafo compilado.")

### 6.2 Visualização

Incluímos a figura para facilitar a leitura do caminho percorrido entre os agentes e a ferramenta.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

## 7. Execução da revisão

### 7.1 Função `gerar_revisao`

Reunimos a execução e a apresentação em `gerar_revisao` para manter simples a chamada pela interface.
A validação do formato permite informar quando o Avaliador não entrega os blocos esperados.

O gabarito fica em um Accordion inicialmente fechado para que o estudante tente responder antes de consultar as soluções.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from IPython.display import display, Markdown
import ipywidgets as widgets

def gerar_revisao(conteudo: str, prioridade: str, foco: str, profundidade: str, tamanho_revisao: str):

    mensagem = f"""
    Conteúdo estudado: {conteudo}
    Prioridade: {prioridade}
    Foco: {foco}
    Profundidade: {profundidade}
    Tamanho da revisão: {tamanho_revisao}
    """

    resultado = app.invoke(
        {"messages": [HumanMessage(content=mensagem)]},
        config={"recursion_limit": 10}
    )

     # Pega apenas respostas geradas pelos agentes
    respostas_agentes = [
        msg for msg in resultado["messages"]
        if isinstance(msg, AIMessage) and msg.content
    ]

    revisao = respostas_agentes[0].content
    saida_avaliador = respostas_agentes[-1].content

    if "<QUESTOES>" not in saida_avaliador or "<GABARITO>" not in saida_avaliador:
        raise ValueError("O Avaliador não retornou o formato esperado.")

    questoes = (
        saida_avaliador
        .split("<QUESTOES>")[1]
        .split("</QUESTOES>")[0]
        .strip()
    )

    gabarito = (
        saida_avaliador
        .split("<GABARITO>")[1]
        .split("</GABARITO>")[0]
        .strip()
    )

    display(Markdown("## 📚 Revisão\n" + revisao))
    display(Markdown("## 📝 Questões\n" + questoes))

    saida_gabarito = widgets.Output()
    with saida_gabarito:
        display(Markdown(gabarito))

    accordion = widgets.Accordion(children=[saida_gabarito])
    accordion.set_title(0, "🔐 Mostrar gabarito")

    accordion.selected_index = None # Começa fechado
    display(accordion)


    return {
        "revisao": revisao,
        "questoes": questoes,
        "gabarito": gabarito
    }

### 7.2 Teste integrado com OpenMP

Este exemplo verifica o fluxo completo, desde a revisão até a apresentação das questões e do gabarito.
Como a chamada direta também retorna os resultados, o gabarito pode aparecer na saída deste teste; na interface final, ele é apresentado no Accordion.

In [ ]:
gerar_revisao(
    conteudo="OpenMP: parallel for, static, dynamic e guided",
    prioridade="dynamic e guided",
    foco="comparacao",
    profundidade="intermediaria",
    tamanho_revisao="curta"
)

## 8. Interface de uso

Mantivemos a interação com `input()` pela simplicidade de uso no notebook. Os widgets ficam restritos à apresentação do gabarito.

### 8.1 Seleção das opções

As escolhas são numeradas e validadas para permitir que o usuário corrija erros de digitação sem interromper a execução.

In [ ]:
def escolher_opcao(titulo, opcoes, nomes_exibicao=None):
    print(f"\n{titulo}")

    for i, opcao in enumerate(opcoes, start=1):
        nome = nomes_exibicao.get(opcao, opcao) if nomes_exibicao else opcao
        print(f"{i}. {nome}")

    while True:
        escolha = input("Digite o número da opção: ").strip()

        if escolha.isdigit():
            indice = int(escolha) - 1

            if 0 <= indice < len(opcoes):
                return opcoes[indice]

        print("Opção inválida. Tente novamente.")


# Nomes mais bonitos para mostrar ao usuário
nomes_focos = {
    "conceitual": "Conceitual",
    "comparacao": "Comparação",
    "aplicacao": "Aplicação",
    "interpretacao": "Interpretação",
    "codigo": "Código",
    "resolucao_de_problemas": "Resolução de problemas"
}

### 8.2 Execução interativa

Após informar o conteúdo, o usuário escolhe prioridade, foco, profundidade e tamanho. A revisão é gerada em seguida, sem precisar fazer outra chamada manual.

In [ ]:
# 1. Conteúdo estudado
conteudo = input("Descreva o conteúdo que você estudou: ").strip()

while not conteudo:
    print("O conteúdo não pode ficar vazio.")
    conteudo = input("Descreva o conteúdo que você estudou: ").strip()


# 2. LLM analisa o conteúdo
print("\nAnalisando o conteúdo...")

opcoes = gerar_opcoes(conteudo)

print("Conteúdo analisado!")


# 3. Prioridade
prioridade = escolher_opcao(
    "Qual tópico você quer priorizar?",
    opcoes["topicos"]
)


# 4. Foco
foco = escolher_opcao(
    "Qual deve ser o foco da revisão?",
    opcoes["focos"],
    nomes_focos
)


# 5. Profundidade
profundidade = escolher_opcao(
    "Qual deve ser a profundidade?",
    [
        "basica",
        "intermediaria",
        "avancada"
    ],
    {
        "basica": "Básica",
        "intermediaria": "Intermediária",
        "avancada": "Avançada"
    }
)


# 6. Tamanho da revisão
tamanho_revisao = escolher_opcao(
    "Qual deve ser o tamanho da revisão?",
    [
        "curta",
        "media",
        "detalhada"
    ],
    {
        "curta": "Curta",
        "media": "Média",
        "detalhada": "Detalhada"
    }
)


# 7. Executa o sistema multiagente
print("\nGerando sua revisão...\n")

resultado = gerar_revisao(
    conteudo=conteudo,
    prioridade=prioridade,
    foco=foco,
    profundidade=profundidade,
    tamanho_revisao=tamanho_revisao
)

## 9. Exemplos de teste

Os exemplos abaixo usam preferências fixas, como no teste com OpenMP, para conferir as revisões em assuntos diferentes sem preencher a interface.

### 9.1 História — Revolução Francesa

Teste com foco na interpretação das causas da revolução e questões de nível básico.

In [ ]:
gerar_revisao(
    conteudo="Revolução Francesa: sociedade estamental, crise econômica, Iluminismo e queda do Antigo Regime",
    prioridade="sociedade estamental e crise econômica",
    foco="interpretacao",
    profundidade="basica",
    tamanho_revisao="curta"
)

### 9.2 Biologia — Divisão celular

Teste para comparar mitose e meiose, com revisão média e questões de nível intermediário.

In [ ]:
gerar_revisao(
    conteudo="Divisão celular: mitose, meiose, número de cromossomos e variabilidade genética",
    prioridade="diferenças entre mitose e meiose",
    foco="comparacao",
    profundidade="intermediaria",
    tamanho_revisao="media"
)

### 9.3 Matemática — Função quadrática

Teste com foco na resolução de problemas, combinando revisão detalhada e questões avançadas.

In [ ]:
gerar_revisao(
    conteudo="Função quadrática: raízes, discriminante, vértice da parábola e problemas de máximos e mínimos",
    prioridade="vértice da parábola e problemas de máximos e mínimos",
    foco="resolucao_de_problemas",
    profundidade="avancada",
    tamanho_revisao="detalhada"
)

## 10. Considerações finais

Os resultados dependem da qualidade do conteúdo informado e das respostas do modelo, que podem conter erros.
Uma extensão possível seria corrigir as respostas do estudante, mas deixamos essa tarefa fora do escopo para manter o projeto concentrado na criação da revisão e da avaliação.